[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-10-polars-backends.ipynb#scrollTo=ii000001)

---
# Day 10 · Polars & Multiple DataFrame Backends
**certified-journeys / hamilton-certified** · Day 10 · Backends

> **Goal for today:** Port the Day 4 feature pipeline from pandas to Polars with minimal code changes, use `@check_output` to validate Polars schemas, and benchmark both backends.

In [ ]:
%pip install -q 'sf-hamilton[polars]' polars

## Hamilton's Backend Abstraction

Hamilton's core value: **your function logic is backend-agnostic**. The type annotations tell Hamilton what backend each node uses; the Driver config determines which execution path is active.

| What changes between backends | What stays the same |
|---|---|
| Type annotations (`pd.Series` → `pl.Series`) | Function names (node names) |
| Driver config (`backend='polars'`) | Dependency graph structure |
| A few API differences (`.clip` → `.clip`) | Business logic inside functions |
| `@config.when(backend=...)` branches | Tags, decorators, module layout |

For most transformations (arithmetic, comparisons, log, clip), pandas and Polars have near-identical APIs. The porting effort is usually 20–30% of the code, not a rewrite.

In [ ]:
import sys, types, time
import numpy as np
import pandas as pd
import polars as pl
from hamilton import driver
from hamilton.function_modifiers import tag, extract_columns, config, check_output
from hamilton.plugins import h_pandas, h_polars

# Generate synthetic dataset in both formats
rng = np.random.default_rng(10)
N = 50_000  # large enough to show Polars perf advantage

data = {
    'age':            rng.integers(18, 75, N).astype(float),
    'tenure_months':  rng.integers(0, 72, N).astype(float),
    'monthly_spend':  rng.exponential(80, N),
    'num_products':   rng.integers(1, 6, N).astype(float),
    'has_complaint':  rng.integers(0, 2, N).astype(float),
    'days_since_last_login': rng.integers(0, 90, N).astype(float),
}
pandas_df = pd.DataFrame(data)
polars_df = pl.DataFrame(data)

print(f'Dataset: {N:,} rows')
print(f'pandas: {pandas_df.memory_usage(deep=True).sum() / 1024:.0f} KB')
print(f'polars: {polars_df.estimated_size() / 1024:.0f} KB')

## Step 1 · Pandas Pipeline (Baseline)

The Day 4 pipeline, condensed into one module. We'll use this as the reference to port.

In [ ]:
# ── pandas pipeline ────────────────────────────────────────────────────────────

@extract_columns('age','tenure_months','monthly_spend','num_products',
                 'has_complaint','days_since_last_login')
def raw_features__pandas(raw_data: pd.DataFrame) -> pd.DataFrame:
    return raw_data[['age','tenure_months','monthly_spend','num_products',
                     'has_complaint','days_since_last_login']]

@tag(feature_type='numerical')
def age_zscore__pandas(age: pd.Series) -> pd.Series:
    return (age - age.mean()) / age.std()

@tag(feature_type='numerical')
def tenure_years__pandas(tenure_months: pd.Series) -> pd.Series:
    return tenure_months / 12.0

@tag(feature_type='numerical')
def spend_log__pandas(monthly_spend: pd.Series) -> pd.Series:
    return np.log1p(monthly_spend)

@tag(feature_type='numerical')
def recency_score__pandas(days_since_last_login: pd.Series) -> pd.Series:
    return 1.0 - (days_since_last_login / days_since_last_login.max())

@tag(feature_type='boolean')
def is_high_spender__pandas(monthly_spend: pd.Series) -> pd.Series:
    return (monthly_spend > monthly_spend.quantile(0.75)).astype(float)

@tag(feature_type='numerical')
def engagement_score__pandas(
    recency_score__pandas: pd.Series, num_products: pd.Series
) -> pd.Series:
    return recency_score__pandas * (num_products / num_products.max())

pandas_module = types.ModuleType('pandas_pipeline')
for fn in [raw_features__pandas, age_zscore__pandas, tenure_years__pandas,
           spend_log__pandas, recency_score__pandas,
           is_high_spender__pandas, engagement_score__pandas]:
    setattr(pandas_module, fn.__name__, fn)
sys.modules['pandas_pipeline'] = pandas_module
print('pandas pipeline module defined')

## Step 2 · Polars Pipeline — Minimal Code Changes

Key API differences between pandas and Polars Series:

| Operation | pandas | Polars |
|---|---|---|
| Mean | `s.mean()` | `s.mean()` ✓ same |
| Std | `s.std()` | `s.std()` ✓ same |
| Log | `np.log1p(s)` | `s.log1p()` |
| Clip | `s.clip(lower=0)` | `s.clip(lower_bound=0)` |
| Quantile | `s.quantile(0.75)` | `s.quantile(0.75)` ✓ same |
| Cast bool→float | `.astype(float)` | `.cast(pl.Float64)` |
| Max | `s.max()` | `s.max()` ✓ same |

In [ ]:
from hamilton.function_modifiers import extract_columns

# ── Polars pipeline — same logic, Polars Series API ───────────────────────────

@extract_columns('age','tenure_months','monthly_spend','num_products',
                 'has_complaint','days_since_last_login')
def raw_features__polars(raw_data: pl.DataFrame) -> pl.DataFrame:
    return raw_data.select(['age','tenure_months','monthly_spend','num_products',
                            'has_complaint','days_since_last_login'])

@tag(feature_type='numerical')
def age_zscore__polars(age: pl.Series) -> pl.Series:
    return (age - age.mean()) / age.std()

@tag(feature_type='numerical')
def tenure_years__polars(tenure_months: pl.Series) -> pl.Series:
    return tenure_months / 12.0

@tag(feature_type='numerical')
def spend_log__polars(monthly_spend: pl.Series) -> pl.Series:
    return (monthly_spend + 1).log(base=float(np.e))  # Polars log1p equivalent

@tag(feature_type='numerical')
def recency_score__polars(days_since_last_login: pl.Series) -> pl.Series:
    return 1.0 - (days_since_last_login / days_since_last_login.max())

@tag(feature_type='boolean')
def is_high_spender__polars(monthly_spend: pl.Series) -> pl.Series:
    return (monthly_spend > monthly_spend.quantile(0.75)).cast(pl.Float64)

@tag(feature_type='numerical')
def engagement_score__polars(
    recency_score__polars: pl.Series, num_products: pl.Series
) -> pl.Series:
    return recency_score__polars * (num_products / num_products.max())

polars_module = types.ModuleType('polars_pipeline')
for fn in [raw_features__polars, age_zscore__polars, tenure_years__polars,
           spend_log__polars, recency_score__polars,
           is_high_spender__polars, engagement_score__polars]:
    setattr(polars_module, fn.__name__, fn)
sys.modules['polars_pipeline'] = polars_module
print('polars pipeline module defined')

## Step 3 · @config.when — One Module, Two Backends

The production pattern: a single module with `@config.when(backend='pandas')` and `@config.when(backend='polars')` variants for each function. The Driver config activates one branch.

In [ ]:
# Config-gated single module — both backends in one file

@config.when(backend='pandas')
def age_zscore__pd(age: pd.Series) -> pd.Series:
    return (age - age.mean()) / age.std()

@config.when(backend='polars')
def age_zscore__pl(age: pl.Series) -> pl.Series:
    return (age - age.mean()) / age.std()

@config.when(backend='pandas')
def spend_log__pd(monthly_spend: pd.Series) -> pd.Series:
    return np.log1p(monthly_spend)

@config.when(backend='polars')
def spend_log__pl(monthly_spend: pl.Series) -> pl.Series:
    return (monthly_spend + 1).log(base=float(np.e))

@config.when(backend='pandas')
def is_high_spender__pd(monthly_spend: pd.Series) -> pd.Series:
    return (monthly_spend > monthly_spend.quantile(0.75)).astype(float)

@config.when(backend='polars')
def is_high_spender__pl(monthly_spend: pl.Series) -> pl.Series:
    return (monthly_spend > monthly_spend.quantile(0.75)).cast(pl.Float64)

@extract_columns('age', 'monthly_spend', 'tenure_months')
@config.when(backend='pandas')
def raw_data__pd(raw_input: pd.DataFrame) -> pd.DataFrame:
    return raw_input[['age', 'monthly_spend', 'tenure_months']]

@extract_columns('age', 'monthly_spend', 'tenure_months')
@config.when(backend='polars')
def raw_data__pl(raw_input: pl.DataFrame) -> pl.DataFrame:
    return raw_input.select(['age', 'monthly_spend', 'tenure_months'])

dual_module = types.ModuleType('dual_backend')
for fn in [age_zscore__pd, age_zscore__pl, spend_log__pd, spend_log__pl,
           is_high_spender__pd, is_high_spender__pl,
           raw_data__pd, raw_data__pl]:
    setattr(dual_module, fn.__name__, fn)
sys.modules['dual_backend'] = dual_module

OUTPUTS = ['age_zscore', 'spend_log', 'is_high_spender']

dr_pd = driver.Builder().with_modules(dual_module).with_config({'backend': 'pandas'}).build()
dr_pl = driver.Builder().with_modules(dual_module).with_config({'backend': 'polars'}).build()

r_pd = dr_pd.execute(OUTPUTS, inputs={'raw_input': pandas_df.head(5)[['age','monthly_spend','tenure_months']]})
r_pl = dr_pl.execute(OUTPUTS, inputs={'raw_input': polars_df.head(5).select(['age','monthly_spend','tenure_months'])})

print('pandas age_zscore:', r_pd['age_zscore'].round(4).tolist())
print('polars age_zscore:', [round(x, 4) for x in r_pl['age_zscore'].to_list()])

### What just happened?
- **One module, two backends** — `@config.when(backend='pandas')` / `@config.when(backend='polars')` activates one set.
- **Identical results** — both produce the same z-scores for the same input data.
- Switching from pandas to Polars is a Driver rebuild with `{'backend': 'polars'}` — no pipeline code changes.

## Step 4 · Benchmark: pandas vs Polars on 50k Rows

In [ ]:
BENCH_OUTPUTS_PD = ['age_zscore__pandas', 'tenure_years__pandas',
                    'spend_log__pandas', 'recency_score__pandas',
                    'is_high_spender__pandas', 'engagement_score__pandas']
BENCH_OUTPUTS_PL = [o.replace('__pandas', '__polars') for o in BENCH_OUTPUTS_PD]

dr_bench_pd = driver.Builder().with_modules(pandas_module).build()
dr_bench_pl = driver.Builder().with_modules(polars_module).build()

RUNS = 5

pd_times, pl_times = [], []
for _ in range(RUNS):
    t0 = time.perf_counter()
    dr_bench_pd.execute(BENCH_OUTPUTS_PD, inputs={'raw_data': pandas_df})
    pd_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    dr_bench_pl.execute(BENCH_OUTPUTS_PL, inputs={'raw_data': polars_df})
    pl_times.append(time.perf_counter() - t0)

pd_avg = sum(pd_times) / RUNS
pl_avg = sum(pl_times) / RUNS

print(f'Dataset size: {N:,} rows, {len(BENCH_OUTPUTS_PD)} features, avg over {RUNS} runs')
print(f'pandas: {pd_avg*1000:.1f} ms')
print(f'polars: {pl_avg*1000:.1f} ms')
print(f'Speedup: {pd_avg/pl_avg:.1f}×  (Polars is faster for this workload)')

### What just happened?
- **Polars is typically 2–5× faster** for transform-heavy workloads on large DataFrames, due to its Rust backend and query optimizer.
- The speedup grows with dataset size — for small DataFrames (<10k rows), the overhead of Polars setup can make pandas faster.
- **Hamilton's abstraction makes the benchmark fair**: identical logic, identical graph structure, only the Series type changed.

## Step 5 · Schema Validation with @check_output on Polars

In [ ]:
from hamilton.function_modifiers import check_output

# @check_output validates the Polars Series dtype and null counts
@check_output(data_type=pl.Series, allow_nans=False)
def age_cleaned_polars(age: pl.Series) -> pl.Series:
    """Clamp age [18, 100] — no nulls allowed in output."""
    return age.clip(lower_bound=18, upper_bound=100).fill_null(30)  # fill with median

@check_output(data_type=pl.Series, allow_nans=False)
def spend_cleaned_polars(monthly_spend: pl.Series) -> pl.Series:
    """Remove negatives — no nulls allowed in output."""
    return monthly_spend.clip(lower_bound=0)

validate_module = types.ModuleType('polars_validate')
validate_module.age_cleaned_polars   = age_cleaned_polars
validate_module.spend_cleaned_polars = spend_cleaned_polars
sys.modules['polars_validate'] = validate_module

dr_v = driver.Builder().with_modules(validate_module).build()
result = dr_v.execute(
    ['age_cleaned_polars', 'spend_cleaned_polars'],
    inputs={
        'age': polars_df['age'],
        'monthly_spend': polars_df['monthly_spend']
    }
)

print('age_cleaned_polars null count:  ', result['age_cleaned_polars'].null_count())
print('spend_cleaned_polars min value: ', result['spend_cleaned_polars'].min())
assert result['age_cleaned_polars'].null_count() == 0
assert result['spend_cleaned_polars'].min() >= 0
print('✓ @check_output passed for both Polars nodes')

### What just happened?
- **`@check_output` works identically with Polars Series** — the decorator is backend-agnostic.
- `allow_nans=False` checks `null_count() == 0` for Polars Series.
- Schema validation is a safety net at the boundary between pipeline stages — catches dtype mismatches early.

In [ ]:
# Challenge: add a Polars variant of `engagement_score` from Day 4
# Port: engagement_score = recency_score * (num_products / num_products.max())
# Use Polars Series API. Add @check_output(data_type=pl.Series, allow_nans=False)
# Benchmark it against the pandas version on 50k rows

# @check_output(data_type=pl.Series, allow_nans=False)
# def engagement_score_polars(recency_score: pl.Series, num_products: pl.Series) -> pl.Series:
#     ...

print('Implement engagement_score_polars with @check_output and benchmark!')

---
## Day 10 key concepts recap

| Concept | What to remember |
|---|---|
| Type annotation = backend signal | `pd.Series` vs `pl.Series` tells Hamilton which backend |
| `@config.when(backend=...)` | Single module, two backends — switch with Driver config |
| API delta | Most operations identical; differences: `.log()`, `.clip(lower_bound=)`, `.cast()` |
| Polars speedup | 2–5× faster on large DataFrames; overhead wins for small |
| `@check_output` | Works the same on Polars Series |

> **Tip:** Hamilton's abstraction means most of your function logic doesn't change when switching backends — only the type annotations and driver config. That's the point of the framework.

---
## What's next
**Day 11** → Lifecycle hooks: add observability to your pipeline without touching function code — log node timing, cache outputs, and validate graph structure at build time.

Mark Day 10 complete in your [tracker](../index.html).